# Лабораторная работа №3
## Метод k-ближайших соседей (KNN)

**Тема:** Обучение модели KNN, подбор гиперпараметров, кросс-валидация

**Цель работы:** Изучение метода k-ближайших соседей, подбор гиперпараметра K с помощью GridSearchCV и RandomizedSearchCV, сравнение качества моделей

**Выполнил:** [ФИО студента]  
**Группа:** [Номер группы]  
**Дата:** [Дата выполнения]

## 1. Описание задания

### Задачи лабораторной работы
1. Выбрать набор данных для задачи классификации или регрессии
2. При необходимости провести удаление/заполнение пропусков и кодирование категориальных признаков
3. Разделить выборку на обучающую и тестовую с помощью `train_test_split`
4. Обучить модель KNN с произвольно заданным гиперпараметром K, оценить качество подходящими метриками
5. Подобрать гиперпараметр K с помощью GridSearchCV и RandomizedSearchCV и кросс-валидации
6. Использовать не менее двух стратегий кросс-валидации
7. Сравнить метрики качества исходной и оптимальной моделей

## 2. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from scipy.stats import randint
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("Библиотеки успешно импортированы")

## 3. Загрузка и предобработка данных

Используем датасет **Titanic** для задачи бинарной классификации (выжил/не выжил). Датасет содержит пропуски и категориальные признаки, требующие предобработки.

In [ ]:
# Загрузка датасета
try:
    import kagglehub
    from pathlib import Path
    path = kagglehub.dataset_download("c/titanic")
    dataset_path = Path(path)
    csv_files = list(dataset_path.glob("*.csv"))
    train_file = next((f for f in csv_files if 'train' in f.name.lower()), csv_files[0])
    df = pd.read_csv(train_file)
    print(f"Датасет загружен: {train_file.name}")
except Exception as e:
    print(f"Загрузка через kagglehub: {e}")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print("Датасет загружен через альтернативный источник")

print(f"\nРазмерность: {df.shape}")
df.head(10)

In [ ]:
# Предобработка: удаляем лишние столбцы, заполняем пропуски, кодируем категории

# Удаляем столбцы, не подходящие для модели
df_clean = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], errors='ignore')

# Заполнение пропусков
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

imputer_num = SimpleImputer(strategy='median')
df_clean[numeric_cols] = imputer_num.fit_transform(df_clean[numeric_cols])

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0] if len(df_clean[col].mode()) > 0 else 'Unknown')

# One-Hot кодирование категориальных признаков
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

# Целевая переменная и признаки
X = df_encoded.drop(columns=['Survived'])
y = df_encoded['Survived']

# Масштабирование (важно для KNN!)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

print("Пропуски после обработки:", X_scaled.isnull().sum().sum())
print(f"Признаков: {X_scaled.shape[1]}, Объектов: {X_scaled.shape[0]}")
X_scaled.head()

## 4. Разделение на обучающую и тестовую выборки

Используем `train_test_split` с фиксированным `random_state` для воспроизводимости.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")
print(f"Распределение классов в train: {dict(y_train.value_counts())}")
print(f"Распределение классов в test: {dict(y_test.value_counts())}")

## 5. Обучение модели KNN с произвольным K

Обучаем модель с K=5 (произвольно выбранное значение) и оцениваем качество.

In [ ]:
K_INITIAL = 5  # Произвольно заданный гиперпараметр

knn_initial = KNeighborsClassifier(n_neighbors=K_INITIAL)
knn_initial.fit(X_train, y_train)
y_pred_initial = knn_initial.predict(X_test)

# Метрики качества для задачи классификации
accuracy_initial = accuracy_score(y_test, y_pred_initial)
precision_initial = precision_score(y_test, y_pred_initial, average='weighted', zero_division=0)
recall_initial = recall_score(y_test, y_pred_initial, average='weighted', zero_division=0)
f1_initial = f1_score(y_test, y_pred_initial, average='weighted', zero_division=0)

print(f"Модель KNN с K={K_INITIAL}")
print("=" * 50)
print(f"Accuracy:  {accuracy_initial:.4f}")
print(f"Precision: {precision_initial:.4f}")
print(f"Recall:    {recall_initial:.4f}")
print(f"F1-score:  {f1_initial:.4f}")
print("\nОтчёт по классам:")
print(classification_report(y_test, y_pred_initial, target_names=['Не выжил', 'Выжил']))

In [ ]:
# Матрица ошибок
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_initial), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Не выжил', 'Выжил'], yticklabels=['Не выжил', 'Выжил'], ax=ax)
ax.set_xlabel('Предсказание')
ax.set_ylabel('Факт')
ax.set_title(f'Матрица ошибок (K={K_INITIAL})')
plt.tight_layout()
plt.show()

## 6. Подбор гиперпараметра K

### 6.1. GridSearchCV

Полный перебор значений K с кросс-валидацией. Используем **StratifiedKFold** (стратифицированная K-Fold) для сохранения баланса классов.

In [ ]:
param_grid = {'n_neighbors': list(range(1, 51)), 'weights': ['uniform', 'distance'], 'p': [1, 2]}

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid,
    cv=cv_stratified,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print("GridSearchCV - лучшие параметры:", grid_search.best_params_)
print(f"Лучший CV score: {grid_search.best_score_:.4f}")

### 6.2. RandomizedSearchCV

Случайный поиск по пространству гиперпараметров. Используем **KFold** как вторую стратегию кросс-валидации.

In [ ]:
param_dist = {
    'n_neighbors': randint(1, 51),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=50,
    cv=cv_kfold,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train, y_train)

print("RandomizedSearchCV - лучшие параметры:", random_search.best_params_)
print(f"Лучший CV score: {random_search.best_score_:.4f}")

### 6.3. Выбор оптимальной модели

Выбираем лучшую модель по результатам поиска (в данном случае — GridSearchCV, т.к. он даёт глобальный оптимум).

In [ ]:
# Используем модель из GridSearchCV (более полный поиск)
knn_optimal = grid_search.best_estimator_
y_pred_optimal = knn_optimal.predict(X_test)

accuracy_optimal = accuracy_score(y_test, y_pred_optimal)
precision_optimal = precision_score(y_test, y_pred_optimal, average='weighted', zero_division=0)
recall_optimal = recall_score(y_test, y_pred_optimal, average='weighted', zero_division=0)
f1_optimal = f1_score(y_test, y_pred_optimal, average='weighted', zero_division=0)

print(f"Оптимальная модель KNN: K={knn_optimal.n_neighbors}, weights={knn_optimal.weights}, p={knn_optimal.p}")
print("=" * 50)
print(f"Accuracy:  {accuracy_optimal:.4f}")
print(f"Precision: {precision_optimal:.4f}")
print(f"Recall:    {recall_optimal:.4f}")
print(f"F1-score:  {f1_optimal:.4f}")
print("\nОтчёт по классам:")
print(classification_report(y_test, y_pred_optimal, target_names=['Не выжил', 'Выжил']))

## 7. Сравнение метрик исходной и оптимальной моделей

In [ ]:
comparison = pd.DataFrame({
    'Метрика': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    f'Исходная (K={K_INITIAL})': [accuracy_initial, precision_initial, recall_initial, f1_initial],
    f'Оптимальная (K={knn_optimal.n_neighbors})': [accuracy_optimal, precision_optimal, recall_optimal, f1_optimal],
    'Разница': [
        accuracy_optimal - accuracy_initial,
        precision_optimal - precision_initial,
        recall_optimal - recall_initial,
        f1_optimal - f1_initial
    ]
})

print("Сравнение качества моделей")
print("=" * 70)
print(comparison.to_string(index=False))

improvement = (accuracy_optimal - accuracy_initial) * 100
print(f"\nУлучшение Accuracy: {improvement:+.2f}%")
if improvement > 0:
    print("Подбор гиперпараметров улучшил качество модели.")
else:
    print("Исходная модель показала сопоставимое или лучшее качество.")

In [ ]:
# Визуальное сравнение метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
initial_vals = [accuracy_initial, precision_initial, recall_initial, f1_initial]
optimal_vals = [accuracy_optimal, precision_optimal, recall_optimal, f1_optimal]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, initial_vals, width, label=f'Исходная (K={K_INITIAL})', color='coral', alpha=0.8)
axes[0].bar(x + width/2, optimal_vals, width, label=f'Оптимальная (K={knn_optimal.n_neighbors})', color='steelblue', alpha=0.8)
axes[0].set_ylabel('Значение')
axes[0].set_title('Сравнение метрик качества')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3, axis='y')

# Матрицы ошибок рядом
ax1 = axes[1]
sns.heatmap(confusion_matrix(y_test, y_pred_optimal), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Не выжил', 'Выжил'], yticklabels=['Не выжил', 'Выжил'], ax=ax1)
ax1.set_xlabel('Предсказание')
ax1.set_ylabel('Факт')
ax1.set_title('Матрица ошибок оптимальной модели')

plt.tight_layout()
plt.show()

## 8. Выводы

В ходе работы:
1. Загружен и предобработан датасет Titanic (заполнение пропусков, кодирование категорий, масштабирование)
2. Выборка разделена на обучающую и тестовую с помощью `train_test_split`
3. Обучена модель KNN с K=5 и оценена метриками Accuracy, Precision, Recall, F1-score
4. Выполнен подбор гиперпараметров с помощью **GridSearchCV** (стратегия StratifiedKFold) и **RandomizedSearchCV** (стратегия KFold)
5. Сравнены метрики исходной и оптимальной моделей

Подбор гиперпараметра K с кросс-валидацией позволяет найти более подходящую конфигурацию модели и, как правило, улучшить качество предсказаний на тестовой выборке.